[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cu-mkp/m-k-manuscript-data/blob/nlp-analysis/allFolios/xml/tl/ms_fr_640_nlp.ipynb)

# Who Does What to What? Entity-Role Extraction in Ms. Fr. 640

This notebook applies dependency-based NLP to the English translation of
[BnF Ms. Fr. 640](https://edition640.makingandknowing.org/), a 16th-century
French craftsman's manuscript digitised and encoded by the
[Making and Knowing Project](https://www.makingandknowing.org/) (Columbia University).

**Research question:** For each semantic entity type tagged in the XML
(materials, tools, animals, professions, ...), what *transitive action verbs*
appear when that entity is the grammatical subject of a clause -- and what
are the direct objects of those verbs?

**Method in brief:**
1. Parse the XML, extracting plain text while tracking the character positions of every semantic tag.
2. Run spaCy dependency parsing sentence by sentence.
3. For every nominal subject (`nsubj` / `nsubjpass`) whose position falls inside a tagged span, record the head verb and its direct object.
4. Filter: transitive verbs only, no copulas, no pronoun objects.

> **Read the Caveats section** at the bottom before drawing conclusions.
> This is an exploratory, first-pass analysis using a small general-purpose model.

## 1 · Setup
Install dependencies and download the spaCy English model. Run once per session.

In [ ]:
%%capture
!pip install spacy lxml wordcloud matplotlib
!python -m spacy download en_core_web_sm

In [ ]:
import re
import math
import urllib.request
from collections import defaultdict

import spacy
from lxml import etree
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from wordcloud import WordCloud

print('All imports OK')

## 2 · Load the manuscript XML

Fetched directly from the public Making and Knowing GitHub repository -- no upload required.

In [ ]:
XML_URL  = "https://raw.githubusercontent.com/cu-mkp/m-k-manuscript-data/master/allFolios/xml/tl/all_tl.xml"
XML_FILE = "all_tl.xml"

urllib.request.urlretrieve(XML_URL, XML_FILE)
print(f"Downloaded {XML_FILE}")

## 3 · Configuration

### Semantic tag vocabulary

The manuscript XML uses a controlled tag vocabulary developed by the Making and Knowing Project.
We track all 17 semantic tags as potential grammatical subjects.
Language tags (`<fr>`, `<la>`, ...) and editorial markup (`<add>`, `<del>`, ...) are excluded -- see Caveats.

In [ ]:
# ---------------------------------------------------------------------------
# Semantic entity tags we want to treat as potential grammatical subjects.
# Each key is the XML tag name; each value is a human-readable label.
#
# We deliberately exclude:
#   - Language tags (fr, la, it, de, es, el, oc, po): these wrap non-English
#     text that the English NLP model cannot parse reliably.
#   - Editorial markup (add, corr, exp, sup, …): these describe how text was
#     written or corrected, not what kind of entity it is.
# ---------------------------------------------------------------------------
TAG_LABELS = {
    'al':  'Animal',
    'bp':  'Body Part',
    'cn':  'Currency',
    'df':  'Definition',
    'env': 'Environment',
    'm':   'Material',
    'md':  'Medical Term',
    'ms':  'Measurement',
    'mu':  'Musical Instrument',
    'pa':  'Plant',
    'pl':  'Place',
    'pn':  'Personal Name',
    'pro': 'Profession',
    'sn':  'Sense Term',
    'tl':  'Tool',
    'tmp': 'Temporal',
    'wp':  'Weapon',
}

# A frozenset (immutable set) of just the tag names, for fast membership tests
SUBJECT_TAGS = frozenset(TAG_LABELS)

# ---------------------------------------------------------------------------
# XML tags whose content is completely removed before NLP processing.
# Including these would introduce noise into the text the NLP model sees.
#
#   comment – editorial note references (empty elements, no text)
#   del     – text the author crossed out; not part of the intended text
#   figure  – graphical objects, no readable text
#   gap     – lacunae (places where text is missing from the manuscript)
#   hr      – horizontal rules (dividers), no text
#   ill     – illegible passages
#   mark    – manuscript symbols like '|' or '/' used as markers
# ---------------------------------------------------------------------------
SKIP_TAGS = frozenset({
    'comment', 'del', 'figure', 'gap', 'hr', 'ill', 'mark',
})

# XML tags replaced by a single space in the extracted text.
# <lb> marks a line break in the manuscript; we treat it as a word separator.
SPACE_TAGS = frozenset({'lb'})

# ---------------------------------------------------------------------------
# Copular (linking) verbs that are excluded from results.
# These verbs connect a subject to a description rather than expressing an
# action, so they do not tell us what the entity *does*.
# Examples: "Colophony IS nothing other than rosin" (identity, not action)
#           "The varnish SEEMS thick" (state, not action)
# ---------------------------------------------------------------------------
COPULA_LEMMAS = frozenset({
    'be', 'seem', 'appear', 'become', 'remain', 'look', 'feel', 'sound',
})

# When a token falls inside more than one tagged span (e.g. <pa> nested
# inside <m>), we need a tiebreaker. <m> (Material) always wins, because
# the manuscript's primary concern is how materials behave.
M_PRIORITY_TAG = 'm'

print(f"{len(TAG_LABELS)} semantic tags tracked")

## 4 · XML text extraction

For each `<ab>` (anonymous text block), we walk the element tree and build:
- a plain-text string suitable for NLP
- a list of `(start, end, text, tag)` spans marking where each semantic tag appears

Character positions are tracked incrementally so multi-word spans are handled correctly.

In [ ]:
def extract_text_and_subject_spans(element):
    """
    Walk a single XML element (typically an <ab> block) and do two things:

    1. Build a plain-text string by concatenating all readable text content,
       skipping or replacing certain tags as configured above.

    2. Record the character positions (start, end) of every element whose
       tag is in SUBJECT_TAGS, along with the element's text and tag name.
       These are called 'spans'.

    Why track character positions?
    --------------------------------
    After extraction, we pass the plain text to spaCy for NLP analysis.
    spaCy returns tokens with character offsets into that plain text.
    By knowing where each tagged span starts and ends in the plain text,
    we can later ask: "is this spaCy token inside a <m> tag?" and answer
    it precisely, even for multi-word tags like <m>turpentine oil</m>.

    How the tree walk works
    -------------------------
    XML elements have two kinds of text:
      - node.text  : text directly inside the opening tag, before any child
      - child.tail : text that follows a child's closing tag, still inside
                     the parent element

    For example: <ab>Take <m>oil</m> and mix</ab>
      ab.text  = "Take "
      m.text   = "oil"
      m.tail   = " and mix"   ← this is the tail of <m>, part of <ab>'s flow

    We process these in order to build the text in reading sequence.

    Returns
    -------
    (raw_text : str, raw_spans : list of (start, end, text, tag))
        'raw' because whitespace has not yet been normalised.
        Positions are character offsets into raw_text.
    """
    buf   = []    # accumulates text fragments; joined at the end
    spans = []    # list of (start, end, text, tag) for each subject element
    pos   = [0]   # running character counter; wrapped in a list so the
                  # nested function 'walk' can modify it (Python closures
                  # cannot reassign variables from the enclosing scope, but
                  # they can mutate mutable objects like lists)

    def walk(node):
        """Recursively process one XML node and all its children."""

        # lxml represents XML comments and processing instructions as nodes
        # with non-string tag values. Skip them.
        tag = node.tag if isinstance(node.tag, str) else None
        if tag is None:
            return

        # If this element is in SKIP_TAGS, ignore it and all its content.
        # Note: the element's 'tail' (text after its closing tag) is added
        # by the *parent's* loop, so we don't lose it here.
        if tag in SKIP_TAGS:
            return

        # If this is a line-break tag, insert a space in the text stream
        # and move on — it has no text content of its own.
        if tag in SPACE_TAGS:
            buf.append(' ')
            pos[0] += 1
            return

        # Check whether this element is one we track as a potential subject.
        is_subject = tag in SUBJECT_TAGS
        if is_subject:
            # Record the position where this tagged span begins
            s_start = pos[0]
            s_tag   = tag

        # Add the element's own text (the part directly after the opening tag)
        if node.text:
            buf.append(node.text)
            pos[0] += len(node.text)

        # Recurse into each child element, then add the child's tail text
        for child in node:
            walk(child)
            # child.tail is the text between this child's closing tag and
            # the next sibling's opening tag — it belongs to the current
            # element's text flow, so we add it here after walking the child.
            if child.tail:
                buf.append(child.tail)
                pos[0] += len(child.tail)

        # If this was a tracked element, record where it ends
        if is_subject:
            s_end  = pos[0]
            # Slice the accumulated buffer to get the exact span text
            s_text = ''.join(buf)[s_start:s_end]
            spans.append((s_start, s_end, s_text.strip(), s_tag))

    walk(element)
    return ''.join(buf), spans


def normalise_and_remap_spans(raw_text, raw_spans):
    """
    The raw text extracted from XML may have irregular whitespace —
    multiple spaces, newlines, tabs — because XML formatting does not
    always reflect the intended reading text. This function:

    1. Collapses any run of whitespace into a single space and strips
       leading/trailing whitespace. This is what we feed to spaCy.

    2. Re-locates each span in the normalised text by searching for its
       (also normalised) text string. We search in document order —
       advancing our search position after each match — so that if the
       same word appears multiple times, we match the right occurrence.

    Why re-locate rather than adjust offsets mathematically?
    ---------------------------------------------------------
    Calculating exactly how whitespace compression shifts each character
    position is fiddly and error-prone. Searching for the span text is
    simpler and reliable for all but pathological edge cases (e.g. the
    exact same multi-word phrase appearing twice in the same block).

    Returns
    -------
    (norm_text : str, norm_spans : list of (start, end, text, tag))
    """
    # Collapse whitespace to single spaces
    norm_text = re.sub(r'\s+', ' ', raw_text).strip()

    norm_spans  = []
    search_from = 0   # keeps track of where to start the next search,
                      # so we match spans in left-to-right document order

    for (_, _, raw_span_text, tag) in raw_spans:
        # Normalise the span text in the same way as the full text
        s_norm = re.sub(r'\s+', ' ', raw_span_text).strip()
        if not s_norm:
            continue   # skip any span that became empty after normalisation

        # Search for this span text starting from the last match position
        idx = norm_text.find(s_norm, search_from)

        # Fallback: if not found from current position (can happen when
        # whitespace normalisation reorders spans), try from the beginning
        if idx == -1:
            idx = norm_text.find(s_norm)

        if idx == -1:
            continue   # span text not found at all — skip it

        end = idx + len(s_norm)
        norm_spans.append((idx, end, s_norm, tag))

        # Advance the search position so the next span is found after this one
        search_from = end

    return norm_text, norm_spans

print('Extraction functions defined')

## 5 · NLP analysis

For every sentence in every `<ab>` block:
1. Find tokens with dependency relation `nsubj` or `nsubjpass`.
2. Check whether the token's character position falls inside a tagged span.
3. If the head verb is transitive (has a direct object child), record the triple.

When tags are nested, `<m>` (Material) takes priority; otherwise the tightest span wins.

In [ ]:
def tightest_subject_span(token, spans):
    """
    Given a spaCy token and a list of tagged spans, return the text and
    tag of the *best* span that contains this token, or None if no span
    contains it.

    "Best" is defined by two priority rules:

    Rule 1 — <m> wins unconditionally
        If any <m> (Material) span contains the token, we return that,
        regardless of any other spans. This handles cases like:
            <m><pa>marshmallow</pa> root</m>
        where 'marshmallow' is inside both a <pa> and an <m>. We want
        to report it as a Material subject, not a Plant subject.

    Rule 2 — tightest span wins among non-<m> tags
        If multiple non-<m> spans contain the token, the shortest one
        wins (innermost/most specific). If lengths are equal, the first
        one in the list wins.

    Parameters
    ----------
    token : spaCy Token
        The token to look up (typically a subject token from the parser).
    spans : list of (start, end, text, tag)
        The tagged spans for the current <ab> block.

    Returns
    -------
    (term_text, tag) if a containing span is found, else None.
    """
    # Character range of the token within the normalised text
    t_start = token.idx
    t_end   = t_start + len(token.text)

    best_m     = None   # best <m> span found: (length, text)
    best_other = None   # best non-<m> span found: (length, text, tag)

    for (ms, me, mt, tag) in spans:
        # Check whether the token falls entirely within this span.
        # (t_start >= ms) means the token starts at or after the span start.
        # (t_end <= me)   means the token ends at or before the span end.
        if t_start >= ms and t_end <= me:
            length = me - ms   # span length in characters

            if tag == M_PRIORITY_TAG:
                # Keep the shortest <m> span (most specific)
                if best_m is None or length < best_m[0]:
                    best_m = (length, mt)
            else:
                # Keep the shortest non-<m> span
                if best_other is None or length < best_other[0]:
                    best_other = (length, mt, tag)

    # Return the best <m> span if one was found; otherwise the best other span
    if best_m is not None:
        return (best_m[1], M_PRIORITY_TAG)
    if best_other is not None:
        return (best_other[1], best_other[2])
    return None


def noun_chunk_for(token, doc):
    """
    Return the full noun-phrase text for a given token.

    spaCy identifies 'noun chunks' — base noun phrases like
    'the colored water' or 'a fine sheen'. The head of such a chunk
    is its main noun. If our object token is the head of a noun chunk,
    we return the whole chunk rather than just the single word, giving
    more informative output.

    Falls back to the token's own text if no chunk is found.

    Parameters
    ----------
    token : spaCy Token  — the direct-object head token
    doc   : spaCy Doc    — the parsed document (needed to iterate chunks)
    """
    for chunk in doc.noun_chunks:
        if chunk.root == token:
            return chunk.text
    return token.text

print('Analysis functions defined')

In [ ]:
print("Loading spaCy model ...")
nlp = spacy.load('en_core_web_sm')

print("Parsing XML ...")
root = etree.parse(XML_FILE).getroot()

rows = []
all_abs = list(root.iter('ab'))
print(f"Processing {len(all_abs):,} <ab> blocks ...")

for i, ab in enumerate(all_abs):
    if i % 500 == 0 and i > 0:
        print(f"  {i:,} / {len(all_abs):,}")

    # Find the enclosing <div> for its id and categories attributes
    div_id = categories = ''
    node = ab.getparent()
    while node is not None:
        if isinstance(node.tag, str) and node.tag == 'div':
            div_id     = node.get('id', '')
            categories = node.get('categories', '')
            break
        node = node.getparent()

    raw_text, raw_spans = extract_text_and_subject_spans(ab)
    if not raw_text.strip() or not raw_spans: continue

    text, spans = normalise_and_remap_spans(raw_text, raw_spans)
    if not text or not spans: continue

    doc = nlp(text)
    for token in doc:
        # Only consider nominal subjects (active and passive)
        if token.dep_ not in ('nsubj', 'nsubjpass'): continue
        match = tightest_subject_span(token, spans)
        if match is None: continue
        subject_term, subject_tag = match

        verb = token.head
        if verb.pos_ not in ('VERB', 'AUX'): continue
        if verb.lemma_ in COPULA_LEMMAS: continue

        # Require a syntactic direct object
        obj_tokens = [c for c in verb.children if c.dep_ in ('obj', 'dobj')]
        if not obj_tokens: continue

        # 300-character context window around the subject token
        window    = 300
        ctx_start = max(0, token.idx - window)
        ctx_end   = min(len(text), token.idx + len(token.text) + window)
        sentence  = text[ctx_start:ctx_end].strip()

        is_passive = token.dep_ == 'nsubjpass'
        for obj_tok in obj_tokens:
            if obj_tok.pos_ == 'PRON': continue     # skip pronoun objects
            obj_text  = noun_chunk_for(obj_tok, doc)
            obj_match = tightest_subject_span(obj_tok, spans)
            rows.append({
                'div_id':        div_id,
                'categories':    categories,
                'subject_tag':   subject_tag,
                'subject_label': TAG_LABELS.get(subject_tag, subject_tag),
                'subject_term':  subject_term,
                'verb_lemma':    verb.lemma_,
                'verb_form':     verb.text,
                'passive':       is_passive,
                'obj_text':      obj_text,
                'obj_tag':       obj_match[1] if obj_match else '',
                'sentence':      sentence,
            })

df = pd.DataFrame(rows)
print(f"\nFound {len(df):,} subject-verb-object triples")

## 6 · Results

### 6.1 · Triples per entity type

In [ ]:
tag_summary = (
    df.groupby(['subject_tag', 'subject_label'])
      .size()
      .reset_index(name='triples')
      .sort_values('triples', ascending=False)
)
tag_summary

### 6.2 · Browse by entity type

Change `TAG` to any tag abbreviation (`m`, `tl`, `pro`, `al`, `wp`, `pl`, ...).

In [ ]:
TAG = 'm'   # <- change this

view = (
    df[df.subject_tag == TAG]
      [['subject_term', 'verb_lemma', 'obj_text', 'passive', 'div_id', 'sentence']]
      .sort_values(['verb_lemma', 'subject_term'])
      .reset_index(drop=True)
)
print(f"{len(view)} triples for <{TAG}> ({TAG_LABELS.get(TAG, TAG)})")
view

### 6.3 · Aggregated summary -- top verb-object pairs per entity type

In [ ]:
summary = (
    df.groupby(['subject_label', 'subject_term', 'verb_lemma', 'obj_text'])
      .size()
      .reset_index(name='count')
      .sort_values(['subject_label', 'count'], ascending=[True, False])
      .reset_index(drop=True)
)
summary[summary['count'] > 1]

### 6.4 · Verbs of desire / necessity

Which entities express need, want, or requirement?

In [ ]:
desire_verbs = ['want', 'need', 'require', 'demand', 'wish']
df[df.verb_lemma.isin(desire_verbs)][['subject_label', 'subject_term', 'verb_lemma', 'obj_text', 'sentence']]

## 7 · Word clouds

One cloud per entity type. Word size reflects verb frequency --
how often entities of that type perform that kind of action.

In [ ]:
TAG_COLORS = {
    'Material':'#1f77b4', 'Profession':'#d62728', 'Tool':'#2ca02c',
    'Weapon':'#ff7f0e', 'Place':'#9467bd', 'Measurement':'#8c564b',
    'Animal':'#e377c2', 'Sense Term':'#7f7f7f', 'Personal Name':'#bcbd22',
    'Plant':'#17becf', 'Environment':'#aec7e8', 'Temporal':'#ffbb78',
    'Definition':'#98df8a',
}

def make_colormap(hex_color):
    r, g, b = int(hex_color[1:3],16), int(hex_color[3:5],16), int(hex_color[5:7],16)
    def color_func(word, font_size, position, orientation, random_state=None, **kw):
        f = 0.55 + random_state.uniform(0, 0.35)
        return (int(r*f), int(g*f), int(b*f))
    return color_func

freq_by_label = (
    df.groupby(['subject_label', 'verb_lemma'])
      .size().reset_index(name='n')
)
labels_with_data = [
    lbl for lbl in freq_by_label.subject_label.unique()
    if freq_by_label[freq_by_label.subject_label == lbl].verb_lemma.nunique() >= 2
]

n, ncols = len(labels_with_data), 2
nrows = math.ceil(n / ncols)
fig = plt.figure(figsize=(18, 5 * nrows))
fig.suptitle(
    "Transitive verb clouds by entity type\n(word size proportional to frequency)",
    fontsize=15, fontweight='bold', y=1.01
)
gs = gridspec.GridSpec(nrows, ncols, figure=fig, hspace=0.45, wspace=0.1)

for idx, label in enumerate(sorted(labels_with_data)):
    subset    = freq_by_label[freq_by_label.subject_label == label]
    freq_dict = dict(zip(subset.verb_lemma, subset.n))
    tag       = df[df.subject_label == label].subject_tag.iloc[0]
    wc = WordCloud(
        width=800, height=380, background_color='white',
        max_words=80, prefer_horizontal=0.85,
        color_func=make_colormap(TAG_COLORS.get(label, '#555555')),
        collocations=False, min_font_size=10,
    ).generate_from_frequencies(freq_dict)
    row, col = divmod(idx, ncols)
    ax = fig.add_subplot(gs[row, col])
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f"{label}  <{tag}>  ({sum(freq_dict.values())} triples)", fontsize=12, fontweight='bold')

for idx in range(n, nrows * ncols):
    row, col = divmod(idx, ncols)
    fig.add_subplot(gs[row, col]).axis('off')

plt.show()

## 8 · Export results

Download the CSVs for use in other tools, or continue to the AI Interpretation section below.

In [ ]:
df.to_csv('subject_verb_obj_detail.csv', index=False)

summary_out = (
    df.groupby(['subject_tag', 'subject_label', 'subject_term', 'verb_lemma'])
      .agg(
          total_count=('obj_text', 'count'),
          objects=('obj_text', lambda x: '; '.join(
              f"{v} ({k})" if k > 1 else v
              for v, k in sorted(pd.Series(x).value_counts().items(), key=lambda t: -t[1])
          ))
      )
      .reset_index()
      .sort_values(['subject_label', 'subject_term', 'verb_lemma'])
)
summary_out.to_csv('subject_verb_obj_summary.csv', index=False)

print('Saved CSVs. Use File -> Download in the Colab sidebar to save them locally.')

## 9 · AI Interpretation with Gemini

Use the Gemini API to generate a scholarly interpretation of the NLP results.

**One-time setup:**
1. Get a free API key at [aistudio.google.com](https://aistudio.google.com) → *Get API key*
2. In this Colab, click the **key icon** in the left sidebar (*Secrets*)
3. Add a secret named `GEMINI_API_KEY` and paste your key
4. Toggle *Notebook access* to ON

Then run the cells below. You can edit the prompts freely to ask different questions.

> **Tip — free tier limits:** The default model is `gemini-1.5-flash-8b`,
> which has the most generous free-tier quota. If you still get a 429 error,
> the `ask()` helper will retry automatically after 60 seconds.
> To use a larger model, change `GEMINI_MODEL` in the setup cell.

In [ ]:
%%capture
!pip install -q google-generativeai

In [ ]:
import time
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# Model choice — free-tier availability varies by account and region.
# If you get a 429 / quota-exceeded error, try the next model down the list:
#   'gemini-1.5-flash-8b'  <- most generous free tier, try this first
#   'gemini-1.5-flash'     <- slightly larger, still free
#   'gemini-2.0-flash-lite'
#   'gemini-2.0-flash'     <- may require billing on some accounts
GEMINI_MODEL = 'gemini-1.5-flash-8b'

_client = genai.GenerativeModel(GEMINI_MODEL)

def ask(prompt, retries=3, wait=60):
    """Send a prompt to Gemini with automatic retry on 429 rate-limit errors."""
    for attempt in range(retries):
        try:
            return _client.generate_content(prompt).text
        except Exception as e:
            if '429' in str(e) and attempt < retries - 1:
                print(f'Rate limit hit — waiting {wait}s before retry {attempt+2}/{retries}...')
                time.sleep(wait)
            else:
                raise
    return None

print(f'Gemini ready ({GEMINI_MODEL})')

In [ ]:
# ---------------------------------------------------------------------------
# Shared data summaries reused across all prompts below.
# We deliberately omit the 'sentence' column — it is long and not needed
# for pattern interpretation (sentences are in the CSVs for manual review).
# ---------------------------------------------------------------------------

# Per-tag triple counts
tag_counts_str = (
    df.groupby(['subject_label', 'subject_tag'])
      .size()
      .reset_index(name='triples')
      .sort_values('triples', ascending=False)
      .to_string(index=False)
)

# Top triples across all tags, capped at 60 rows
top_triples_csv = (
    df.groupby(['subject_label', 'subject_term', 'verb_lemma', 'obj_text'])
      .size()
      .reset_index(name='count')
      .sort_values(['subject_label', 'count'], ascending=[True, False])
      .head(60)
      [['subject_label', 'subject_term', 'verb_lemma', 'obj_text', 'count']]
      .to_csv(index=False)
)

print('Data summaries ready')
print(f'  tag_counts_str : {len(tag_counts_str)} chars')
print(f'  top_triples_csv: {len(top_triples_csv)} chars')

### 9.1 · Overall patterns

Ask Gemini to interpret what the full set of triples reveals about craft knowledge
in the manuscript.

In [ ]:
prompt = f"""
You are a historian of science and technology specialising in early modern craft knowledge.

The table below contains subject-verb-object triples automatically extracted from
BnF Ms. Fr. 640, a 16th-century French craftsman's manuscript (ca. 1580s) written
in Middle French and translated into English. The manuscript covers goldsmithing,
casting, painting, varnishing, medicine, and natural history, among other topics.

Each triple records a semantically tagged entity (Material, Tool, Profession, etc.)
acting as the grammatical subject of a transitive action verb, with its direct object.
The triples were extracted by spaCy dependency parsing — treat them as a first-pass
signal, not ground truth.

Entity type counts:
{tag_counts_str}

Top subject-verb-object triples (top 60 by frequency, grouped by entity type):
{top_triples_csv}

Please interpret what these patterns collectively reveal, addressing:
1. How do **materials** behave or act in the manuscript's language — what do they do?
2. What roles do **professions** (craftsmen, artisans) play as grammatical agents?
3. What does the language around **tools** and **weapons** suggest about their use?
4. Are there any surprising, counterintuitive, or historically significant patterns?
5. What questions do these results open up for further research?

Write 400-600 words in an academic but readable style.
"""

print(ask(prompt))

### 9.2 · Deep dive by entity type

Change `FOCUS_TAG` to zoom in on any entity type.
Valid tags: `m`, `tl`, `pro`, `wp`, `al`, `pl`, `pa`, `pn`, `sn`, `env`, `ms`, `df`, `tmp`

In [ ]:
FOCUS_TAG = 'm'   # <- change this

focus_label = TAG_LABELS.get(FOCUS_TAG, FOCUS_TAG)

# Top 40 triples for this tag only (no sentence column)
focus_csv = (
    df[df.subject_tag == FOCUS_TAG]
      .groupby(['subject_term', 'verb_lemma', 'obj_text', 'passive'])
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
      .head(40)
      .to_csv(index=False)
)

prompt2 = f"""
You are a historian of early modern craft and material culture.

The data below are subject-verb-object triples for **{focus_label}** entities (<{FOCUS_TAG}>)
extracted from BnF Ms. Fr. 640 (16th-century craftsman's manuscript, ca. 1580s).
The 'passive' column is True when the entity is acted upon rather than acting.

{focus_csv}

Analyse what these triples reveal about how {focus_label.lower()}s are described and
used in this manuscript:
- Which {focus_label.lower()}s appear most active (frequent as subjects)?
- What verbs dominate — do they suggest transformation, preservation, application?
- Are passive constructions (passive = True) revealing — what is done *to* these entities?
- What does this language tell us about 16th-century craft knowledge and practice?

Write 300-500 words.
"""

print(ask(prompt2))

### 9.3 · Custom question

Edit `your_question` to ask anything about the data.
The prompt includes the top-60 triples summary (no sentences) as context.

In [ ]:
your_question = """
Which entries suggest the author had hands-on practical experience rather than
book learning? What patterns in the verb choices support your answer?
"""

prompt3 = f"""
You are a historian of early modern craft and technology.

Here are subject-verb-object triples from BnF Ms. Fr. 640 (16th-century craftsman's manuscript).
Entity type counts:
{tag_counts_str}

Top triples (top 60):
{top_triples_csv}

{your_question}
"""

print(ask(prompt3))

---

## 10 · Caveats and methodology notes

### Model accuracy
`en_core_web_sm` is a small statistical model trained on modern English (news, web text).
Ms. Fr. 640 is 16th-century craftsman's prose with archaic syntax, technical vocabulary,
and mixed French-English constructions. Parse accuracy is lower than the ~90% on standard benchmarks.
The `sentence` column lets you verify every triple manually.

### What is and isn't captured

| Included | Excluded |
|---|---|
| Active subjects (`nsubj`) | Copulas: *be, seem, appear, become, remain* |
| Passive subjects (`nsubjpass`) -- flagged in `passive` column | Pronoun objects (*it, them, itself*) -- need coreference resolution |
| Direct objects (`obj` / `dobj`) | Prepositional objects (*acts on X*, *works with X*) |
| All 17 semantic tag types | Language tags (`<fr>`, `<la>`, ...) -- not tracked as subjects |

### Multi-word spans
spaCy assigns `nsubj` to a single head token. For `<m>turpentine oil</m>`,
the head is typically `oil`. Only that token's character position is checked against spans.

### Coordinated subjects
In "Sulfur and vermilion makes the same effect", spaCy marks one as `nsubj`
and the other as `conj` -- only the `nsubj` token is matched.

### Nested tags -- priority rules
When a token falls inside overlapping spans: `<m>` wins over all others;
among non-`<m>` tags, the tightest (innermost) span wins.

### Excluded XML content
`<del>` (struck-through), `<ill>` (illegible), `<gap>` (lacuna), `<comment>`,
`<figure>`, `<hr>`, `<mark>` are stripped. `<lb>` (line break) is replaced with a space.

### Sparse tag types
`<bp>`, `<cn>`, `<md>`, `<mu>` returned zero triples -- these entity types
appear primarily as objects or adjuncts in recipe prose, not as agents of action.

---

*Data: [Making and Knowing Project](https://www.makingandknowing.org/), Columbia University.*
*XML encoding: [ms-transcription schema](https://github.com/cu-mkp/m-k-manuscript-data/tree/master/schema).*